Setting up DLT(Delta Live Table) Scenario Dataset
- setting up datasets and objects for DLT(Delta Live Table).

For working with DLT(Delta Live Table) Pipeline, we need to go with multi-node cluster 
 and also access mode :shared

compute ---> creating multi-node cluster

General
Compute name :Neeraj cluster

Policy :Shared Compute

Performance

Machine learning
Databricks runtime :16.4 LTS (Scala 2.12)
                    Scala 2.12, Spark 3.5.2

Photon acceleration --> disable

Worker type :Standard_DS3_v2 8 GB Memory, 4 Cores
Min :1   Max:1

-> CREATE COMPUTE

here access mode: shared means 
- the result of DLT(Delta Live Table) goes into tables
- to access those tables-->databricks kept a restriction 
that, 

    - we can access those table using shared mode cluster and
                 
    - we cannot get shared mode access in single node cluster
                 
    - so we create multi-node cluster in shared mode


Setting up the catalog and creating External location


In [0]:

--CREATE CATALOG IF NOT EXISTS dev;
--CREATE DATABASE IF NOT EXISTS dev.demodb;

Goto Datalake -> Create one folder in dbfscontainer -> Add directory -> dataset_ch9
and upload 7files into this

creating External Location

In [0]:

--CREATE EXTERNAL LOCATION IF NOT EXISTS `external-data`
--URL  'abfss://dbfscontainer@mystoragelakeadb6pmgroup.dfs.core.windows.net/dataset_ch9'
--WITH (CREDENTIAL `adb6pm-storage-credential`);

Creating external volume

In [0]:

--CREATE EXTERNAL VOLUME IF NOT EXISTS dev.demodb.landing_zone
--LOCATION 'abfss://dbfscontainer@mystoragelakeadb6pmgroup.dfs.core.windows.net/dataset_ch9'

To see the data inside the volume

Creating 2 folders within the landing_zone--->

1) customers  

2) invoices

 and ingesting data

customers folder will be created under landing_zone and its visible at container also.

invoices folder will be created under landing_zone and its visible at container also.

ingesting data

using cp command to copy data from landingzone to cusomers and invoices.

Copy 2021 invoice file.

Copy 2022 invoice file.

first from landing zone ingest data into 2 tables

1) customers_raw
2) invoices_raw

we want to do it using auto-loader.

I want DLT pipeline only to create table and load data.

DLT pipeline code can be written in 2 ways

1) using sparksql
2) using python


1. Using Sparksql:

Create your bronze layer tables and ingesting data from landing zone.

here we want DLT (Delta Live Table) to create and fill table using incremental approach (Streaming).

here we want to create a streaming table and
next we tell DLT (Delta Live Table) from where to get the data from 

get from cloudfiles() function

- cloudfiles() is our auto loader

in cloudfiles() function, we provide parameters to define autoloader configurations
and landingzone directory

1st paramter is landingzone directory from where cloudfiles() should start
ingesting data

ex: cloud_files('Volumes/dev/demodb/landing_zone/customers'

2nd parameter--->cloud file format

3rd paramter---->autoloader configuration.

by default it comes with default configuration

and other additional configurations we can provide.

Create STREAMING TABLE customers_raw

In [0]:

CREATE OR REFRESH STREAMING TABLE customers_raw
AS SELECT *,current_timestamp() as load_time
from cloud_files('/Volumes/dev/demodb/landing_zone/customers','csv',map("cloudFiles.inferColumnTypes","true"))

STREAMING TABLE invoices_raw

In [0]:

CREATE OR REFRESH STREAMING TABLE invoices_raw
AS SELECT *,current_timestamp() as load_time
from cloud_files('/Volumes/dev/demodb/landing_zone/invoices','csv',map("cloudFiles.inferColumnTypes","true"))

Next step is reading data from customers_raw and apply some data quality check
to filter out poor qaulity data and fill data into customers_cleaned

Now writing the DLT (Delta Live Table) code.

for writing the DLT code,we have 2 parts

1st create table 

and then, fill data into table


all tables in silver layer are streaming tables

customer_cleaned is a incremental table or streaming table


Expectations:
- In DLT (Delta Live Table) --> data quality checks are performed through expectations

- expectation are like constraints
- these constraints in DLT are called as expectations i.e Data quality expectation

syntax for definig expectations:
- constraintname expect(condition)

ex: customer   id is NULL------->constraint failed

customer   id is not null--->constraint passed

- what to do if constarint passed
- what to do if constraint failed

Now 2nd part is ----> filling data into table 

bringing only specific columns and renaming

- we have created customer_raw ---> it is a streaming table -> create by DLT pipeline

- where can we find this table--->in which schema/database ?

- tables which are created within DLT pipeline by DLT itself,
- it can refer by using a keyword called live

- live ---> intermediate/temporary schema->created by DLT where all the files/tables can be found
- as mytable is a streaming table, i will wrap it in STREAM()

In [0]:

CREATE OR REFRESH STREAMING TABLE customers_cleaned(
    CONSTRAINT valid_customer EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW    
)AS SELECT CustomerID as customer_id, CustomerName as customer_name, load_time 
 from STREAM(live.customers_raw)

- customer_raw table created within DLT pipeline,
so i can refer it using --------> live.customers_raw

- as it is incremental table or streaming table
and i need to read it as a streaming table
- i need to read incremental data from this table not the entire table
- that's why we give stream()
- if we wont write stream(),it will read the entire table.

Next ---> creating invoices_cleaned

here also 2 steps:
1) Creating table
2) Filling data into the table

In [0]:

CREATE OR REFRESH STREAMING TABLE invoices_cleaned(
  CONSTRAINT valid_invoice_and_qty EXPECT (invoice_no IS NOT NULL AND quantity >0) ON VIOLATION DROP ROW)
  PARTITIONED BY (invoice_year,country)
  AS
  SELECT InvoiceNo as invoice_no,
         StockCode as stock_code,
         Description	as description,
         Quantity as quantity,
        to_date(InvoiceDate,"d-M-y H.m") as invoice_date,
        UnitPrice	as unit_price,
        CustomerID	as customer_id,
        Country	as country,
        year(to_date(InvoiceDate,"d-M-y H.m")) as invoice_year,
        month(to_date(InvoiceDate,"d-M-y H.m")) as invoice_month,
        load_time
  FROM STREAM(live.invoices_raw)

live.invoices_raw ---> here if i wont give live
it searches in default database
but it is not in default database
- it is created by DLT pipeline
- so refer it using Live
- I doesnt want to refer entire table
- I just want to read incremental data-->
- so iam using stream() -> so it reads only incremental data

- so this is intermediate table or high quality data table for silver layer

Next step is----> creating customers table and invoices table 

- here customer table should implement SCD type-2

- if update customer information comes-->we need to implement SCD type-2

- what is SCD type-2?
  
  - The old record will be closed and marked as in-active with timestamp and
  
  - new updated information about the customer will also get inserted into
  - customer table and it it will be time stamped as active

ex: 15th July----->old record was active

16th July----->New updated information has arrived.
- Customer information is active from 16th July onwards using the new record

- so we implement this SCD-Type2


Step 3: Build your SCD Type2 dimensions using CDC from silver layer

There are 2parts in DLT stmt:
  1) create the target table --->creating incremental(streaming)table
  2) Fill the data into the target table

 here we want to implement SCD-Type2 ---> it is done using merge stmt.

 - duplicates removal using merge stmt

 - if record already available ----> update
 - if record doesnt exist ------> insert that record
  this is what merge stmt does.

 - In DLT,applying merge stmt comes with diff syntaxes

- apply changes to ---> target table
- from stream(sourcetable) --> we want only incremental data

next is joining condition 
keys(customer_id) ---> here unique key is single column key.

writing merge using DLT approach

most powerful feature of DLT -----> sequenceBy ---> load time.

means, if we have recieved 2 updates for the same customer: 
ex: 1st update at 10am  and

2nd update at 11am

so we will have 2 updates and target is 1 record,
one of this two, we need to take for building SCD-Type2.

- implement it by using 1st rec and then

- implement it  by using 2nd rec.

If there are duplicate records for the same customerId, sort them by using loadtime and then implement SCD-Type2, thats what sequenceBy will do.

next is,
- we need to say that we are implementing SCD Type-2
- that can be said as ---> stored as SCD Type-2.

In [0]:

CREATE OR REFRESH STREAMING TABLE customers;
APPLY CHANGES INTO live.customers
FROM STREAM(live.customers_cleaned)
KEYS(customer_id)
SEQUENCE BY load_time
STORED AS SCD TYPE 2;
-- command is only allowed to be run in DLT pipelines;

2 stmts seperated by semicolon

Next step is ----> creating invoices table ---> No duplicates ---> latest information

using merge stmt ---> using change Data capture(CDC)
- if record found--->update
- if not found ----->insert


here again 2 steps--->
1) create table
2) filling data into table--> provide details how to fill data into table---> merge stmt

process:

apply changes ------> to target

from --------> source

keys()------> specify keys for matches

invoice no --> for each invoice ---> we get multiple records.
ex: in 1 invoice ---> we get 3 records
- so invoice no is not unique key. 
- so we need to use composite key for matching composite key ---> multiple keys so we use ---> 'invoice' and 'date' also as composite key.

In [0]:

CREATE OR REFRESH STREAMING TABLE invoices PARTITIONED BY (invoice_year,country);
APPLY CHANGES INTO live.invoices
FROM STREAM(live.invoices_cleaned)
KEYS(invoice_no,stock_code,invoice_date)
SEQUENCE BY load_time;
-- command is only allowed to be run in DLT pipelines;

what happens if 2 updates arrived for same invoice

one update arrived at 9:15am
 "    "       "    at 9:18am

do you want to implement 9:15 one first

and

implement 9:18 second ?

but if you are implementing 9:18 first and then
9:15 second then, we are overriding the new information with the old information.

so sequence is important here

so whenever we implement merge ---> sequence by --> loadtime --> is important

you need to have some field which identifies the order in which transaction happened

so here we use 'loadtime' field.

generally your records will have created  timestamp,we will use that for sequencing.

since my input dataset doesnt come with created timestamp so we have created 'loadtime' filed to do that.

here we can know which record is loaded first and then second.


we implemented-Bronze layer.
"       "      Silver layer.

The last step is creating Gold layer table to refresh itself.

Full refresh means what??

Full refresh ---> also known as "full reset" ,which involves completely re-building the target table or view from scratch. 

Means it involves:

- Data Truncate: all existing data in target table or view is removed.
- State reset: state information associated with streaming tables and flows are cleared.
- Full reprocessing: All data from the source is re-processed from the beginning to recreate the target table or view.

Creating Gold layer.

You have 2 options here
- you can create a view: view will always refreshed when we query/use it.

but view has a disadvantage
- whenever we touch the view, the query is fired
and it will go and query the base tables
    
- views are typically slow.

What is the alternative approach

in databases/DWH's ----> we have materialized views
- Materialized views ---> It is filled with data
--> It is refreshed if there is any change in the source table. 
- If we refresh the materialized view : the customers/consumers can query.

Full refresh in DLT is done by implementing materialized view

In last step, we want to implement 'daily_sales' materialized view.

upto now, we were creating streaming tables --> incremental tables.

but now we are implementing materialized view

Here also 2 steps:
 - create/refresh the table
 - filling the table

Keyword in DLT ---> for creating mterialized view is 'Live'. 

here 'Live' represent materialized view
- if we want to build incremental table ---> go with streaming table.

- if we want materialized view with fullrefresh,
whenever change in source ---> we say live.

In [0]:

CREATE OR REFRESH LIVE TABLE daily_sales
as select country,invoice_year,invoice_month,invoice_date,
round(sum(quantity*unit_price),2) as total_sales
from LIVE.invoices
WHERE invoice_year=2022 and country="United Kingdom"
GROUP BY country,invoice_year,invoice_month,invoice_date

Gold Layer

Materialized view: Daily_Sales table.